# Other embedders

In [ ]:
import lancedb
import pandas as pd

db = lancedb.connect("vector_db")

df = pd.read_json("data/jokes.json")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [15]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)

class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField()

print(f"{model.ndims() = }")

table_jokes = db.create_table("jokes_sentence_transformers", schema = JokeModel)
table_jokes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 791.14it/s]


model.ndims() = 384


LanceTable(name='jokes_sentence_transformers', _conn=LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_db'))

In [17]:
table_jokes.search("math_jokes").to_pandas().head()

,joke,vector,_distance


## Custom embedder

In [ ]:
model = (
    get_registry()
    .get("nvidia")
    .create(name="nvidia/nemotron-3.5-embed-8b", device="cpu")
)

In [23]:
from custom_embeddings import nvidia_nemotron

model = get_registry().get("nvidia-nemotron").create()

class JokeModel(LanceModel):
    jokes: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField()

model.ndims()

4096

In [25]:
table = db.create_table("jokes_nvidia", schema=JokeModel, mode="overwrite")
table.add(df)

AddResult(version=3)

In [26]:
table.search("ge mig ett programmingsskämt").limit(5).to_pandas()

,jokes,vector,_distance
0,The C# compiler walked into a bar. The bartend...,"[-0.005577, -0.054626, 0.017075, -0.009445, 0....",1.140816
1,Why did the Python programmer get bitten? Beca...,"[-0.013855, -0.042114, 0.011337, -0.019638, 0....",1.150602
2,Why did the C# developer go broke? He kept usi...,"[-0.010506, -0.058746, 0.014175, -0.005405, 0....",1.228449
3,Why is C# like a musical? It has so many classes.,"[-0.00077, -0.090088, 0.015549, 0.007221, 0.01...",1.318060
4,C# programmers love coffee—they're always work...,"[-0.00407, -0.055267, 0.009743, 0.003815, 0.00...",1.347130
